# Cursul 6. Analiza exploratorie
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

Datele sunt curate (C4) și în baza de date (C5). Înainte de orice model, **analiza exploratorie** caută ce merită verificat. Nu este o listă de grafice, ci un ciclu: o întrebare, un grafic sau un tabel care îi răspunde, apoi întrebarea nouă pe care o ridică răspunsul ([Wickham și colegii, *R for Data Science*](https://r4ds.hadley.nz/eda.html)).

Caietul urmează cinci întrebări, valabile pe orice date:

| Întrebarea | Pe datele SCI |
|---|---|
| **1. Ce conțin datele?** | numărul de perechi, o sursă externă, extremele, lipsurile |
| **2. Cum arată o valoare obișnuită?** | distribuția SCI, scara logaritmică |
| **3. Ce variază împreună?** | patru ipoteze: distanța, granița, economia, migrația |
| **4. Ce iese din tipar?** | perechile mult mai legate decât distanța le-ar da |
| **5. Ce grupuri apar?** | comunitățile rețelei |

Fiecare secțiune se încheie cu o **decizie** pentru modelul din C7. La final, aceleași cinci întrebări formează fișa EDA a echipei, pe datele proiectului.

## 1. Ce conțin datele?
Conexiunea este cea din C5; se citesc cele trei tabele de analiză. Înainte de primul grafic, trei verificări care nu cer nicio ipoteză ([Peng, *Exploratory Data Analysis Checklist*](https://bookdown.org/rdpeng/exdata/exploratory-data-analysis-checklist.html)).

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from sqlalchemy import create_engine

engine = create_engine("sqlite:///data/dsw.db")    # baza din C5

regiuni = pd.read_sql("select * from regiuni", engine)
perechi = pd.read_sql("select * from perechi_regiuni", engine)
tari = pd.read_sql("select * from perechi_tari", engine)
len(regiuni), len(perechi), len(tari)

**Numărul de rânduri, față de o socoteală.** 298 de regiuni, fiecare cu toate celelalte, o singură dată: 298 × 297 / 2. Un join greșit sau un filtru uitat se vede întâi aici.

In [ ]:
len(perechi), 298 * 297 // 2

**Un total, față de o sursă externă.** Populația României, adunată pe regiuni, trebuie să aibă ordinul de mărime cunoscut: în jur de 19 milioane. O unitate greșită (mii în loc de persoane) s-ar vedea imediat.

In [ ]:
regiuni[regiuni["tara"] == "RO"]["populatie"].sum()

**Extremele, citite una câte una.** Cele mai legate și cele mai puțin legate perechi.

In [ ]:
perechi.nlargest(3, "sci")[["regiune_1", "regiune_2", "sci"]]

In [ ]:
perechi.nsmallest(3, "sci")[["regiune_1", "regiune_2", "sci"]]

**Interpretează:** ce regiune apare la ambele capete? Rândul ei din tabelul regiunilor:

In [ ]:
regiuni[regiuni["regiune"] == "NO0B"]

Svalbard are câteva mii de locuitori, dar populația din tabel este 0. SCI este împărțit la numărul de utilizatori, iar o regiune aproape goală dă rapoarte extreme, în ambele sensuri. Extremele nu sunt aici descoperiri, ci o consecință a felului în care este construit indicatorul.

**Decizie pentru C7:** Svalbard se exclude din model. În caietul de azi rămâne, ca rezultatele să fie comparabile cu cele din C5.

### Lipsurile: câte și unde
Numărul de valori lipsă spune puțin; contează unde sunt.

In [ ]:
regiuni.isna().sum()

In [ ]:
regiuni[regiuni["retele_sociale"].isna()]["tara"].value_counts().head()

Pe hartă se vede mai bine. Contururile regiunilor vin de la GISCO, ca nomenclatorul din C3; `gpd.read_file` le citește direct de la adresă, iar indicatorii se lipesc după cod, ca în C4.

In [ ]:
harta = gpd.read_file("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_3035_LEVL_2.geojson")
harta = harta.rename(columns={"NUTS_ID": "regiune"})[["regiune", "geometry"]]
harta = harta.merge(regiuni, on="regiune")                      # contururile și indicatorii

harta["lipsa"] = harta.isna().sum(axis=1)                       # câte variabile lipsesc în fiecare regiune
ax = harta.plot(column="lipsa", cmap="Reds", legend=True, figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Variabile lipsă, pe regiuni")
ax.set_axis_off()

**Interpretează:** lipsurile sunt la întâmplare sau pe țări întregi? Consecința: o comparație care folosește PIB-ul pierde Elveția și Norvegia, una care folosește rețelele sociale pierde Germania și Franța. **Decizie:** la fiecare rezultat se spune pe câte perechi a fost calculat.

## 2. Cum arată o valoare obișnuită?
**Anticipează:** media SCI va fi mai mare sau mai mică decât mediana?

In [ ]:
perechi["sci"].describe().round(0)

In [ ]:
ax = perechi["sci"].hist(bins=40)
ax.set(title="SCI, scara originală", xlabel="SCI", ylabel="perechi");

In [ ]:
ax = np.log10(perechi["sci"]).hist(bins=40)
ax.set(title="Aceleași date, scara log10", xlabel="log10(SCI)", ylabel="perechi");

Câteva perechi foarte legate trag media în sus. Pe scara logaritmică (2 înseamnă 100, 3 înseamnă 1.000) distribuția se vede. **Decizie:** SCI se citește pe scara logaritmică, iar comparațiile folosesc mediana.

## 3. Ce variază împreună?
Fără ipoteze, orice pereche de variabile ar fi de comparat. [Bailey și colegii (2020)](https://doi.org/10.1007/978-3-030-60975-7_1) găsesc pentru Europa efecte puternice ale distanței și granițelor; [Bailey și colegii (2018)](https://doi.org/10.1257/jep.32.3.259) arată că SCI este asociat cu migrația. De aici, patru așteptări, scrise înainte de date:

| Ipoteza | Ce ar trebui să se vadă | Cum se verifică |
|---|---|---|
| **H1. Distanța** | SCI scade cu distanța | mediane pe intervale de distanță, Spearman |
| **H2. Granița** | la aceeași distanță, regiunile din aceeași țară sunt mai legate | aceeași țară față de țări diferite, pe intervale |
| **H3. Economia** | regiunile asemănătoare economic sunt mai legate | decalajul de PIB și SCI |
| **H4. Migrația** | țările cu mulți migranți între ele sunt mai legate | migranții, ca număr și ca rată |

„Verificare” înseamnă aici căutarea unor tipare compatibile sau nu cu așteptarea, nu demonstrarea unor cauze.

### 3.1. H1: distanța
Distanța dintre două regiuni se ia între centrele lor. Harta este în proiecția europeană EPSG:3035, care măsoară în metri, deci distanța se calculează direct. Centrul geometric nu este capitala și nici centrul populației, dar ajunge pentru explorare.

In [ ]:
centre = harta.set_index("regiune").centroid           # centrul fiecărei regiuni

capat_1 = centre[perechi["regiune_1"]].values           # centrul primului capăt al fiecărei perechi
capat_2 = centre[perechi["regiune_2"]].values           # centrul celui de-al doilea
perechi["distanta_km"] = capat_1.distance(capat_2) / 1000
perechi[["regiune_1", "regiune_2", "distanta_km", "sci"]].head()

In [ ]:
puncte = perechi.sample(5000, random_state=42)          # 5.000 de perechi, ca punctele să nu se acopere
ax = puncte.plot.scatter(x="distanta_km", y="sci", logx=True, logy=True, alpha=0.15, s=6)
ax.set(xlabel="Distanța (km, log)", ylabel="SCI (log)", title="H1: 5.000 de perechi");

Graficul desenează 5.000 de perechi; calculele folosesc toate perechile. **Spearman** compară ordinea valorilor, între -1 și 1, și nu este tras de câteva valori uriașe: negativ înseamnă că perechile mai îndepărtate tind să aibă SCI mai mic.

In [ ]:
perechi["distanta_km"].corr(perechi["sci"], method="spearman")

Mediana SCI pe intervale de distanță arată forma relației mai bine decât un singur număr. `pd.cut` pune fiecare distanță în intervalul ei; apoi se agregă, ca în C5.

In [ ]:
praguri = [0, 100, 250, 500, 1000, 2000, 10000]
perechi["interval_km"] = pd.cut(perechi["distanta_km"], praguri)
perechi.groupby("interval_km", observed=True)["sci"].agg(["size", "median"])

### 3.2. H2: granița, la aceeași distanță
SCI-ul mediu este mult mai mare în aceeași țară, dar perechile interne sunt și mai apropiate: comparația ar amesteca granița cu distanța. Comparația se face deci **în grupuri comparabile**, pe același interval de distanță.

In [ ]:
perechi["aceeasi_tara"] = perechi["tara_1"] == perechi["tara_2"]
h2 = perechi.groupby(["interval_km", "aceeasi_tara"], observed=True)["sci"].median().unstack()   # intervale × (da, nu)
h2

In [ ]:
ax = h2.plot.bar(logy=True, rot=30)
ax.set(xlabel="Distanța (km)", ylabel="SCI median (log)", title="H2: la aceeași distanță");

**Interpretează:** de câte ori este mai mare mediana internă decât cea externă, la 250-500 km? Se păstrează diferența la toate distanțele? Nu este încă efectul cauzal al graniței: limba și istoria rămân amestecate. **Decizie:** distanța și granița intră în model ca variabile separate.

### 3.3. H3: asemănarea economică
Perechea nu are sens (A-B este tot una cu B-A), deci se folosește **diferența absolută** de PIB, doar pe perechile cu PIB la ambele capete.

In [ ]:
economice = perechi.dropna(subset=["pib_locuitor_1", "pib_locuitor_2"])     # PIB la ambele capete
decalaj = (economice["pib_locuitor_1"] - economice["pib_locuitor_2"]).abs()   # diferența, fără semn
decalaj.corr(economice["sci"], method="spearman")

In [ ]:
len(economice), len(perechi)

**Interpretează:** semnul este cel așteptat? Bailey și colegii (2020) găsesc că decalajele mai mari pot merge cu legături mai puternice, iar migrația este o explicație posibilă: oamenii pleacă din regiunile mai sărace spre cele mai bogate.

### 3.4. H4: migrația
Perechile de țări cu migrația cunoscută în ambele sensuri.

In [ ]:
cu_migratie = tari.dropna(subset=["migranti"])          # migrația cunoscută în ambele sensuri

ax = cu_migratie.plot.scatter(x="migranti", y="sci", logx=True, logy=True, alpha=0.6)
ax.set(xlabel="Migranți în ambele sensuri (log)", ylabel="SCI (log)", title="H4: migrație și legături sociale");

In [ ]:
cu_migratie["migranti"].corr(cu_migratie["sci"], method="spearman")

Corelația este pozitivă, dar nu foarte mare. SCI este însă **relativ**, împărțit la numărul de utilizatori din cele două țări: un milion de migranți înseamnă mult pentru Austria și puțin pentru Germania. Comparabilă cu el este o **rată**: migranți la 1.000 de locuitori ai celor două țări, cu populația de la Banca Mondială, din C2.

In [ ]:
bm = pd.read_csv("https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/raw/bm.csv",
                 keep_default_na=False, na_values=[""])                                   # din C2; NA = Namibia
pop = bm[(bm["indicator"] == "populatie") & (bm["an"] == 2024)].set_index("tara")["valoare"]   # țară -> populație

populatia_perechii = cu_migratie["tara_1"].map(pop) + cu_migratie["tara_2"].map(pop)
cu_migratie["la_mie"] = 1000 * cu_migratie["migranti"] / populatia_perechii
cu_migratie["la_mie"].corr(cu_migratie["sci"], method="spearman")

**Interpretează:** cât crește corelația? **Decizie:** în model intră rata, nu numărul.

**Exercițiu.** Zborurile spun cât se circulă. Corelația Spearman dintre `pasageri` și `sci`, pe perechile care au zboruri: este mai mare sau mai mică decât la migrație?

In [ ]:
# 1. Perechile de țări care au pasageri (dropna pe coloana pasageri)
# 2. Corelația Spearman dintre pasageri și sci, ca la migrație

## 4. Ce iese din tipar?
Distanța singură explică o mare parte din SCI. Pentru a vedea ce iese din tipar, fiecare pereche se compară cu perechile aflate **la aceeași distanță**: SCI-ul ei împărțit la mediana intervalului ei. Peste 1 înseamnă mai legată decât ar sugera distanța; sub 1, mai puțin. Este o primă formă a reziduului din modelul C7.

`transform("median")` calculează mediana fiecărui interval și o pune pe fiecare rând al lui.

In [ ]:
mediana_intervalului = perechi.groupby("interval_km", observed=True)["sci"].transform("median")
perechi["fata_de_distanta"] = perechi["sci"] / mediana_intervalului
perechi.nlargest(5, "fata_de_distanta")[["regiune_1", "regiune_2", "distanta_km", "sci", "fata_de_distanta"]]

**Interpretează:** primele cinci perechi au toate Svalbard la un capăt. Decizia din secțiunea 1 se confirmă: o regiune aproape nelocuită domină orice raport.

### Partenerii unei regiuni, peste așteptare
Pentru o regiune, partenerii din alte țări cei mai legați **față de distanță**. Regiunea poate fi pe oricare capăt al perechii, deci se iau ambele cazuri și se pun unul sub altul.

In [ ]:
REGIUNE = "RO11"        # Nord-Vest; de încercat și "LU00" (Luxemburg), "PL91" (Varșovia)

ca_1 = perechi[perechi["regiune_1"] == REGIUNE].rename(columns={"regiune_2": "partener", "tara_2": "tara"})
ca_2 = perechi[perechi["regiune_2"] == REGIUNE].rename(columns={"regiune_1": "partener", "tara_1": "tara"})
legaturi = pd.concat([ca_1, ca_2])[["partener", "tara", "distanta_km", "sci", "fata_de_distanta"]]

externi = legaturi[legaturi["tara"] != REGIUNE[:2]].nlargest(10, "fata_de_distanta")   # alte țări, primii 10
externi

**Anticipează:** în ce țări vor fi partenerii? Notează înainte de hartă. Liniile unesc centrele regiunilor; nu sunt drumuri.

In [ ]:
ax = harta.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
for partener in externi["partener"]:                               # o linie spre fiecare partener
    ax.plot([centre[REGIUNE].x, centre[partener].x], [centre[REGIUNE].y, centre[partener].y], color="#c0392b")
harta[harta["regiune"] == REGIUNE].plot(ax=ax, color="#c0392b")
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title=f"{REGIUNE}: partenerii peste așteptare")
ax.set_axis_off()

**Interpretează:** partenerii sunt vecinii sau țările migrației? **Decizie:** perechile care ies din tipar (Spania, Irlanda, Portugalia) se urmăresc în reziduurile modelului.

### Ghicitoarea din C1
La C1, Italia, țara cu cei mai mulți români, era abia pe locul cinci după SCI. Perechile României, cu migranții și rata lor:

In [ ]:
cu_ro = cu_migratie[(cu_migratie["tara_1"] == "RO") | (cu_migratie["tara_2"] == "RO")]
cu_ro.sort_values("sci", ascending=False)[["tara_1", "tara_2", "sci", "migranti", "la_mie", "pasageri"]].head(8)

Italia are cea mai mare rată de migranți, dar Austria, Germania, Ungaria și Irlanda au SCI mai mare. Migrația explică mult, dar nu tot. Ungaria este un indiciu: puțini migranți, dar SCI mare, pentru că o parte din legături vin din minoritatea maghiară din România.

## 5. Ce grupuri apar?
Aceleași perechi, privite împreună, formează o **rețea**: fiecare regiune este un **nod**, fiecare pereche o **legătură**, cu SCI ca **pondere**.

In [ ]:
G = nx.from_pandas_edgelist(perechi, "regiune_1", "regiune_2", edge_attr="sci")
G.number_of_nodes(), G.number_of_edges()

O **comunitate** este un grup de regiuni mai legate între ele decât cu restul. Algoritmul [Louvain](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.louvain.louvain_communities.html) le caută doar din ponderi: nu știe nimic despre țări sau hartă. `seed` fixează punctul de pornire, ca rezultatul să fie același la fiecare rulare.

In [ ]:
grupuri = nx.community.louvain_communities(G, weight="sci", seed=1)   # o listă de grupuri de regiuni

comunitate = {}
for numar, grup in enumerate(grupuri):
    for regiune in grup:
        comunitate[regiune] = numar          # regiune -> numărul grupului ei
harta["comunitate"] = harta["regiune"].map(comunitate)
len(grupuri)

In [ ]:
ax = harta.plot(column="comunitate", categorical=True, cmap="tab20", figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Comunitățile rețelei SCI")
ax.set_axis_off()

Culorile sunt doar etichete: comunitatea 3 nu este „mai mult” decât comunitatea 2. **Interpretează:** comunitățile urmează granițele? Una care trece de o graniță este aceeași excepție ca în secțiunea 4? Gruparea singură nu spune de ce (migrație, limbă, istorie); pentru asta trebuie alte date.

## Deciziile pentru model
| Ce s-a văzut | Decizia pentru C7 |
|---|---|
| SCI are o coadă lungă | scara logaritmică și mediana |
| Svalbard: populația 0, extreme în ambele sensuri | se exclude |
| PIB lipsește în Elveția, Norvegia și Balcanii de vest | H3 pe perechile cu PIB, spus explicit |
| distanța și granița, ambele clare | două variabile separate |
| migrația contează ca rată | rata la mie, nu numărul |
| Spania, Irlanda, Portugalia, Ungaria | excepțiile de urmărit în reziduuri |

## Fișa EDA a echipei
Aceleași cinci întrebări, pe datele proiectului. Fiecare răspuns are un număr și spune pe câte cazuri a fost calculat; fișa intră în `README.md`.

| Întrebarea | Ce se raportează |
|---|---|
| 1. Ce conțin datele? | câte rânduri și câte ar trebui să fie; un total față de o sursă externă; ce lipsește și unde |
| 2. Cum arată o valoare obișnuită? | distribuția variabilei principale, scara, extremele citite una câte una |
| 3. Ce variază împreună? | o ipoteză din literatură, verificată în grupuri comparabile |
| 4. Ce iese din tipar? | două cazuri mult peste sau sub așteptare și o explicație posibilă |
| 5. Ce grupuri apar? | o hartă sau o grupare, cu o excepție care merită explicată |

Prima trecere, pe orice tabel: se înlocuiește `regiuni` cu tabelul echipei.

In [ ]:
tabel = regiuni                 # tabelul echipei, citit din baza ei

print(tabel.shape)              # câte rânduri și câte coloane
print(tabel.isna().sum())       # ce lipsește, pe coloane
tabel.describe().round(1)       # cum arată fiecare variabilă numerică

**Exercițiul echipei.** Perechea coridorului, în `cu_migratie`: SCI, migranții, rata și pasagerii. Este peste sau sub perechile cu o rată de migrație asemănătoare?

In [ ]:
# Perechea coridorului în cu_migratie (tara_1 și tara_2, în ordine alfabetică)

## Ce duce analiza mai departe
Cinci întrebări, fiecare cu o decizie. În C7, variabilele (distanța, granița, rata migranților) intră în același model, iar ce rămâne neexplicat se compară cu excepțiile găsite aici.

**Se poate explica fără cod?** De ce se verifică numărul de rânduri înainte de grafice? De ce extremele trebuie citite una câte una? De ce SCI se citește pe scara logaritmică? De ce granița se compară la aceeași distanță? Ce înseamnă o pereche „peste așteptare”?